# Lab 7 - Votre premier Agent Analyste de Données

**Navigation** : [Lab 6 <<](../Lab6-First-Agent/Lab6-First-Agent.ipynb) | [Index](../../README.md) | [>> Lab 8](../../../../AgenticDataScience/Day4-Foundations/Lab8-ADK-Introduction.ipynb)

## Objectifs d'apprentissage

A la fin de ce laboratoire, vous saurez :
1. Creer un agent specialise avec LangChain pour l'analyse de donnees
2. Utiliser `create_pandas_dataframe_agent` pour donner des capacites d'execution de code a un agent
3. Dialoguer en langage naturel avec un agent pour obtenir des insights
4. Comprendre comment un LLM raisonne pour utiliser des outils (code Python/Pandas)

### Prerequis
- Python 3.10+
- Cle API OpenAI configuree (`OPENAI_API_KEY`)
- Connaissance de Pandas (Lab 4 complete)
- Connaissance de LangChain (Lab 2 complete)

### Duree estimee : 45-60 minutes

Nous allons maintenant fusionner les deux mondes : la Data Science (Pandas) et l'IA Agentique (LangChain). L'objectif est de créer un agent qui peut effectuer des analyses de données à notre place.

- **Étape 1 :** Préparation de l'Environnement et des Données


In [1]:
import pandas as pd
from langchain_openai import ChatOpenAI
from langchain_experimental.agents.agent_toolkits import create_pandas_dataframe_agent
import os

# Assurez-vous d'avoir votre clé API OpenAI dans vos variables d'environnement
# Par exemple:
# os.environ["OPENAI_API_KEY"] = "sk-..."

df = pd.read_csv('../Lab4-DataWrangling/transactions.csv')

print("Imports OK : pandas, langchain_openai, langchain_experimental")
print(f"Donnees chargees : {len(df)} lignes, {len(df.columns)} colonnes")

Imports OK : pandas, langchain_openai, langchain_experimental


Un agent, comme un humain, a besoin de données de qualité pour bien travailler. Nous allons donc répéter les mêmes étapes de nettoyage que dans le Lab 4.

In [2]:
# Supprimer les lignes où la quantité est manquante car une transaction sans quantité n'est pas exploitable
df.dropna(subset=['quantite'], inplace=True)

# Remplir le prix manquant avec la moyenne des prix du même produit (stratégie d'imputation)
df['prix_unitaire'] = df.groupby('id_produit')['prix_unitaire'].transform(lambda x: x.fillna(x.mean()))

# Convertir la colonne 'date' en datetime avec format='mixed' pour gérer les variations
df['date'] = pd.to_datetime(df['date'], format='mixed', errors='coerce')

# Créer une colonne 'chiffre_affaires' pour faciliter les calculs
df['chiffre_affaires'] = df['quantite'] * df['prix_unitaire']

print("Données nettoyées et prêtes pour l'agent. Aperçu :")
df.head()


Données nettoyées et prêtes pour l'agent. Aperçu :


,date,id_produit,categorie,quantite,prix_unitaire,chiffre_affaires
0,2023-10-01,A101,Electronique,5.0,120.0,600.0
1,2023-10-01,B202,Livre,10.0,15.5,155.0
2,2023-10-02,A101,Electronique,3.0,120.0,360.0
3,2023-10-03,C303,Maison,2.0,75.9,151.8
4,2023-10-03,B202,Livre,6.0,15.5,93.0


- **Étape 2 :** Le Cerveau et le nouvel "Outil"


Voici la fonction clé : `create_pandas_dataframe_agent`. C'est un "constructeur d'agent" spécialisé. Au lieu de lui donner nos propres outils manuellement, cette fonction va analyser le DataFrame et fournir à l'agent une boîte à outils (`PythonREPLTool`) lui permettant d'écrire et d'exécuter du code Python (et donc Pandas) pour répondre à des questions.

> **Référence — CodeAct.** Le mécanisme au cœur de cet agent — un LLM qui *écrit puis exécute lui-même du code* (ici via `PythonREPLTool`) pour agir et observer le résultat — est le paradigme **CodeAct** (*executable code actions*), formalisé par Wang et al. (2024) : utiliser du code exécutable comme espace d'action unifie raisonnement, appel d'outils et calcul dans une même boucle, plutôt que d'émettre des appels d'API figés. C'est précisément ce que fait l'agent pandas : il génère du code Pandas, l'exécute, lit la sortie, puis itère. Le paradigme ReAct (boucle Pensée→Action→Observation, Lab 6) sous-tend cette itération ; le prolongement data-science SOTA (DS-STAR) est au Lab 10.

In [1]:
llm = ChatOpenAI(temperature=0, model="gpt-4o")

# Création de l'agent
# Note: agent_type="openai-tools" est une méthode moderne et robuste qui utilise les "OpenAI Tools" pour une meilleure fiabilité.
# allow_dangerous_code=True est requis pour autoriser l'exécution de code Python dans l'agent
agent_executor = create_pandas_dataframe_agent(llm, df, agent_type="openai-tools", verbose=True, allow_dangerous_code=True)

print("Agent LangChain configure : LLM gpt-4o, agent_type=openai-tools, DataFrame transactions")

Agent LangChain configure : LLM gpt-4o, agent_type=openai-tools, DataFrame transactions


- **Étape 3 :** Dialoguer avec l'Agent Analyste


In [4]:
agent_executor.invoke({"input": "Combien y a-t-il de lignes dans les données ?"})



> Entering new AgentExecutor chain...



Invoking: `python_repl_ast` with `{'query': 'len(df)'}`


6

Il y a 6 lignes dans les données.

> Finished chain.


{'input': 'Combien y a-t-il de lignes dans les données ?',
 'output': 'Il y a 6 lignes dans les données.'}

### Question de calcul : Agregation metier

Testons maintenant une question qui necessite un calcul d'agregation. L'agent devra identifier la colonne `chiffre_affaires` et appliquer la fonction `.sum()` appropriee. Le parametre `verbose=True` nous permettra de voir le raisonnement de l'agent.


In [5]:
agent_executor.invoke({"input": "Quel est le chiffre d'affaires total réalisé ?"})



> Entering new AgentExecutor chain...



Invoking: `python_repl_ast` with `{'query': "df['chiffre_affaires'].sum()"}`


1839.8

Le chiffre d'affaires total réalisé est de 1839.8.

> Finished chain.


{'input': "Quel est le chiffre d'affaires total réalisé ?",
 'output': "Le chiffre d'affaires total réalisé est de 1839.8."}

### Question d'exploration : Valeurs uniques

Explorons maintenant les categories de produits disponibles. Cette question teste la capacite de l'agent a identifier et utiliser les methodes appropriees de Pandas (comme `.unique()` ou `.nunique()`).


In [6]:
agent_executor.invoke({"input": "Liste les catégories de produits uniques."})



> Entering new AgentExecutor chain...



Invoking: `python_repl_ast` with `{'query': "df['categorie'].unique().tolist()"}`


['Electronique', 'Livre', 'Maison']

Les catégories de produits uniques sont : Electronique, Livre, et Maison.

> Finished chain.


{'input': 'Liste les catégories de produits uniques.',
 'output': 'Les catégories de produits uniques sont : Electronique, Livre, et Maison.'}

### Question complexe : Analyse comparative

Maintenant, posons une question qui necessite un raisonnement plus elabore. L'agent devra non seulement calculer des agregations, mais egalement comparer et identifier le maximum. Observez comment il decompose le probleme en etapes logiques.


In [7]:
agent_executor.invoke({"input": "Quelle est la catégorie qui a généré le plus de chiffre d'affaires ?"})



> Entering new AgentExecutor chain...



Invoking: `python_repl_ast` with `{'query': "df.groupby('categorie')['chiffre_affaires'].sum().idxmax()"}`


Electronique

La catégorie qui a généré le plus de chiffre d'affaires est "Electronique".

> Finished chain.


{'input': "Quelle est la catégorie qui a généré le plus de chiffre d'affaires ?",
 'output': 'La catégorie qui a généré le plus de chiffre d\'affaires est "Electronique".'}

## 🎯 Exercices Pratiques

### Exercice 1 : Questions à l'agent

Formulez et testez 3 questions supplémentaires à poser à l'agent `agent_executor` :
1. Une question de filtrage (ex: transactions avec quantité > 5)
2. Une question d'analyse temporelle (ex: chiffre d'affaires par date)
3. Une question complexe de votre choix (ex: produit le plus rentable, tendances, etc.)

**Analysez la sortie verbose** : Observez comment l'agent décompose chaque question en étapes Python exécutables.

**Indices** : Utilisez `agent_executor.invoke({"input": "votre question"})` pour chaque question.

In [1]:
# Votre code pour l'Exercice 1

# TODO: Formulez et testez vos 3 questions
# Question 1 - Filtrage
reponse_filtre = None

# Question 2 - Analyse temporelle
reponse_temporelle = None

# Question 3 - Question complexe de votre choix
reponse_complexe = None

# Note: Les réponses seront affichées automatiquement par le mode verbose de l'agent
print("Exercice a completer")

Exercice a completer


### Exercice 2 : Comparer les performances de deux modeles LLM

Tous les LLM ne se valent pas en termes de capacite d'analyse de donnees. Vous allez comparer les performances du modele `gpt-4o` (deja utilise) avec un modele plus leger comme `gpt-3.5-turbo` sur une meme question analytique.

**Objectif** : Creer un second agent avec un modele different, lui poser la meme question complexe, et comparer la qualite des reponses.

**Indices** :
- Creez un nouvel agent avec `ChatOpenAI(model="gpt-3.5-turbo", temperature=0)` a la place de `gpt-4o`
- Utilisez la meme fonction `create_pandas_dataframe_agent()` avec le nouveau LLM
- Posez une question analytique complexe aux deux agents (ex: "Quelle est la correlation entre la quantite vendue et le chiffre d'affaires ?")
- Comparez la pertinence et la precision des deux reponses

In [8]:
# Exercice 2 : Comparaison de modeles LLM
# Creez un second agent avec un modele different et comparez les resultats

# Etape 1: Creez un LLM avec un modele different
# Indice: llm_light = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
llm_light = None  # Remplacez None

# Etape 2: Creez un nouvel agent avec ce LLM
# Indice: create_pandas_dataframe_agent(llm_light, df, agent_type="openai-tools", verbose=True, allow_dangerous_code=True)
agent_light = None  # Remplacez None

# Etape 3: Posez la meme question complexe aux deux agents
question_test = "Quelle est la categorie avec le prix unitaire moyen le plus eleve et quel est ce prix ?"

# Reponse de l'agent gpt-4o (deja existant)
# reponse_4o = agent_executor.invoke({"input": question_test})

# Reponse de l'agent gpt-3.5-turbo
# reponse_light = agent_light.invoke({"input": question_test})

# Etape 4: Comparez les reponses
# print("--- Reponse gpt-4o ---")
# print(reponse_4o["output"])
# print("\n--- Reponse gpt-3.5-turbo ---")
# print(reponse_light["output"])

print("Exercice 2 a completer : comparaison de deux modeles LLM")

Exercice a completer


### Exercice 3 : Analyser un nouveau dataset avec l'agent

Un analyste de donnees doit souvent travailler avec des sources de donnees nouvelles. Vous allez charger un dataset different (les donnees des etudiants creees dans le notebook Pandas) et demander a l'agent de produire une analyse complete.

**Objectif** : Creer un nouveau DataFrame a partir de donnees synthetiques et configurer un agent pour l'analyser.

**Indices** :
- Creez un DataFrame avec `pd.DataFrame()` contenant des donnees d'etudiants (Nom, Age, Note, Filiere)
- Utilisez `create_pandas_dataframe_agent()` avec le nouveau DataFrame
- Demandez a l'agent de calculer la moyenne par filiere, d'identifier les outliers et de produire un resume

In [9]:
# Exercice 3 : Analyse d'un nouveau dataset
# Creez un DataFrame d'etudiants et demandez a l'agent de l'analyser

# Etape 1: Creez un DataFrame synthetique
# Indice: utilisez pd.DataFrame() avec un dictionnaire de listes
df_etudiants = pd.DataFrame({
    'Nom': ['Alice', 'Bob', 'Charlie', 'Diana', 'Eve', 'Frank', 'Grace', 'Hugo'],
    'Age': [20, 22, 19, 21, 23, 20, 22, 21],
    'Note': [16.5, 12.0, 18.5, 14.0, 9.5, 15.0, 17.5, 11.0],
    'Filiere': ['Info', 'Maths', 'Info', 'Physique', 'Maths', 'Info', 'Physique', 'Maths']
})

# Etape 2: Creez un agent analyste pour ce DataFrame
# Indice: create_pandas_dataframe_agent(llm, df_etudiants, agent_type="openai-tools", verbose=True, allow_dangerous_code=True)
agent_etudiants = None  # Remplacez None

# Etape 3: Posez des questions analytiques a l'agent
# Question 1: Moyenne des notes par filiere
# reponse_moyenne = agent_etudiants.invoke({"input": "Quelle est la moyenne des notes par filiere ?"})

# Question 2: Identification des outliers
# reponse_outliers = agent_etudiants.invoke({"input": "Y a-t-il des etudiants dont la note est significativement differente de la moyenne ?"})

# Question 3: Resume global
# reponse_resume = agent_etudiants.invoke({"input": "Donne un resume statistique complet de ce dataset"})

print("Exercice 3 a completer : analyse d'un nouveau dataset avec l'agent")

Exercice a completer


## Conclusion

Félicitations ! Vous avez créé un véritable agent spécialisé, un "Data Analyst junior".

Le concept clé à retenir est que nous n'avons pas codé les analyses nous-mêmes. Nous avons donné au LLM un **outil** (la capacité d'exécuter du code sur un DataFrame) et un **contexte** (le DataFrame lui-même), et il a **raisonné** pour déterminer comment utiliser cet outil pour atteindre l'objectif.

Dans les prochains jours, nous apprendrons à construire des agents encore plus complexes, avec de multiples outils et des capacités de planification plus avancées.

## References

1. X. Wang, Y. Chen, L. Yuan, et al., *Executable Code Actions Elicit Better LLM Agents* (CodeAct), ICML 2024, PMLR 235:50208-50232, arXiv:2402.01030. Paradigme de l'action-par-code-exécutable : un LLM écrit et exécute lui-même du code pour raisonner et agir — mécanisme central du `create_pandas_dataframe_agent` (Étape 2).
2. S. Yao et al., *ReAct: Synergizing Reasoning and Acting in Language Models*, ICLR 2023, arXiv:2210.03629. Boucle Pensée→Action→Observation sous-jacente à l'itération de l'agent — détaillée au Lab 6.
3. Google Research, *DS-STAR: A State-of-the-Art Versatile Data Science Agent*, 2025. Agent data-science SOTA — référence détaillée au Lab 10.
4. H. Chase, *LangChain*, 2022 ; `langchain_experimental`. Implémentation de `create_pandas_dataframe_agent` / `PythonREPLTool` — référence bibliographique détaillée au Lab 8.
